# Phase 2 — GPT-5.x Model Eval
**FSQA** | fine-tune pipeline

> **Bring your own data.** No data ships with this repo. This notebook reads `./local-data/phase1_fsqa_eval/<run id>/questions.jsonl` (JSON Lines, one question per line, as Phase 1 writes it), plus `rejected.jsonl` in the same folder if it exists. With `QUESTION_RUN_ID = None` it uses the last run folder in name order. Run Phase 1 first, or supply your own file in that format. Each question record needs `id`, `question`, `reference_answer`, `domain`, `difficulty`, and `citation` (a string) or `citations` (a list); rows in `rejected.jsonl` also need `seed_section`.

Scores the GPT-5.x family on the Phase 1 FSQA question set to pick the
generator for Phase 4 training-pair synthesis.

**Design**
- Sweep: 5.2 → 5.6 Sol, every effort level each model actually accepts
- Effort support is **discovered by probe**, not assumed
- Dual judge: Gemini 3.1 Pro + Kimi K3 (cross-family, and cross-hemisphere).
  Agreement rate tells you whether a ranking is real or a judge artifact.
- Every call logs `is_byok` so you know whether your own key paid

**Judges deliberately exclude GPT and Claude** — GPT grading GPT is biased,
and Claude is left out because of a provider data-retention setting.

---
## Notebook setup
1. Secret `your-modal-secret` → `OPENROUTER_API_KEY`
2. Volume `<MODAL_VOLUME_NAME>` → `./local-data`

Run cell 8 (discovery) before anything expensive. It costs pennies and
tells you exactly which configs exist.

In [ ]:
import concurrent.futures
import json
import os
import random
import re
import statistics
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone

import requests

print("imports ok", flush=True)

## Config

In [ ]:
# --- Run mode -------------------------------------------------------------
SMOKE_TEST = False     # True = 3 configs x 10 questions. False = everything.

# --- Candidate models (OpenRouter slugs, verified against the provider page)
CANDIDATE_MODELS = [
    "openai/gpt-5.2",
    "openai/gpt-5.4",
    "openai/gpt-5.4-mini",
    "openai/gpt-5.4-nano",
    "openai/gpt-5.5",
    "openai/gpt-5.6-luna",
    "openai/gpt-5.6-terra",
    "openai/gpt-5.6-sol",
]

# Effort levels to probe. Discovery finds which each model accepts.
CANDIDATE_EFFORTS = ["minimal", "low", "medium", "high", "xhigh"]

# Pro variants are the same weights served with reasoning.mode=pro.
# Cheap to add as extra configs on the top tier only.
PRO_VARIANTS = ["openai/gpt-5.6-sol", "openai/gpt-5.6-terra", "openai/gpt-5.6-luna"]
INCLUDE_PRO = True

# --- Judges ---------------------------------------------------------------
# Gemini via Google; Kimi via DigitalOcean.
JUDGES = {
    "gemini": {
        "slug": "google/gemini-3.1-pro-preview",
        "provider": {"order": ["google-vertex", "google-ai-studio"], "allow_fallbacks": True},
    },
    "kimi": {
        "slug": "moonshotai/kimi-k3",
        "provider": {"order": ["digitalocean"], "allow_fallbacks": False},
    },
}
USE_JUDGES = ["gemini", "kimi"]     # drop one to halve judging cost

# --- Provider routing for the models under test ---------------------------
# Pinned to the OpenAI endpoint with no fallback.
EVAL_PROVIDER = {"order": ["openai"], "allow_fallbacks": False}

# --- Paths ----------------------------------------------------------------
VOLUME_MOUNT = "./local-data"
QUESTIONS_SUBDIR = "phase1_fsqa_eval"
OUTPUT_SUBDIR = "phase2_model_eval"
QUESTION_RUN_ID = None   # None = use most recent Phase 1 run

# --- Execution ------------------------------------------------------------
MAX_WORKERS = 8
REQUEST_TIMEOUT = 300
MAX_RETRIES = 3
SEED = 20260824

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY missing. Attach 'your-modal-secret'.")
if not os.path.isdir(VOLUME_MOUNT):
    raise RuntimeError(f"{VOLUME_MOUNT} not mounted.")

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "X-OpenRouter-Title": "labs-benchmark",
}

random.seed(SEED)

# --- Spend guard -----------------------------------------------------------
# Every OpenRouter call is reserved at its worst case before it is sent and settled after (guarded_post).
# A call is refused, and the run stops, if its model has no entry in PRICE, if its body has no max_tokens,
# or if its worst case would take the run past RUN_BUDGET_USD. PRICE holds each model's listed price in
# USD per million tokens (input, output). Entries the repo does not state are left for the operator to fill.
from datetime import datetime, timezone
import math, os


# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_positive_float(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


RUN_BUDGET_USD = _env_positive_float("RUN_BUDGET_USD", 25.00)   # one cap per run; override with the RUN_BUDGET_USD env var
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
# Confirmed listed prices, USD per million tokens (input, output), checked 2026-11-05. Each is the highest price
# that can apply to a plain request (priority tier, regional endpoint or over-272k band), so reservations are
# conservative. Sources: https://openrouter.ai/api/v1/models/<slug>/endpoints for gpt-5.2, gpt-5.5, gpt-5.6-luna,
# gpt-5.6-sol, gemini-3.1-pro-preview and kimi-k3; https://openrouter.ai/openai/<slug> for gpt-5.4, gpt-5.4-mini,
# gpt-5.4-nano and gpt-5.6-terra. A model with no entry here is refused before any paid call.
PRICE = {
    "openai/gpt-5.2": (1.75, 14),
    "openai/gpt-5.4": (2.75, 16.5),
    "openai/gpt-5.4-mini": (0.825, 4.95),
    "openai/gpt-5.4-nano": (0.22, 1.375),
    "openai/gpt-5.5": (25, 112.5),
    "openai/gpt-5.6-luna": (0.8, 3.6),
    "openai/gpt-5.6-terra": (2.2, 13.2),
    "openai/gpt-5.6-sol": (8.8, 33),
    "google/gemini-3.1-pro-preview": (7.2, 32.4),
    "moonshotai/kimi-k3": (4.5, 22.5),
}
missing_prices = sorted(set(CANDIDATE_MODELS + (PRO_VARIANTS if INCLUDE_PRO else []) + [JUDGES[j]["slug"] for j in USE_JUDGES]) - set(PRICE))
if missing_prices:
    raise RuntimeError(f"Refusing to run: no confirmed listed price for {missing_prices}. Look up each model's "
                       "input and output price (USD per million tokens) on its OpenRouter endpoints page, add it "
                       "to PRICE in this cell, then rerun.")
# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
# One ledger root for every notebook, from _ledger_root() above: SPEND_LEDGER_DIR if set, else <repo root>/spend_ledger.
# Run ledgers live at <root>/<RUN_ID>/spend_ledger.jsonl.
SPEND_ROOT = _ledger_root()
LEDGER_DIR = os.path.join(SPEND_ROOT, RUN_ID)
os.makedirs(LEDGER_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(LEDGER_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks. A single exclusive lock file, SPEND_ROOT/runner.lock, shared by
# every notebook: a second runner refuses to start while it exists. It is removed on a clean exit only. A stale
# lock (a runner that died without exiting) is never cleared automatically: confirm no runner is alive, then
# delete SPEND_ROOT/runner.lock by hand to resume.
import atexit
RUNNER_LOCK = os.path.join(SPEND_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner may be active, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)


print(f"Models under test : {len(CANDIDATE_MODELS)}")
print(f"Efforts to probe  : {CANDIDATE_EFFORTS}")
print(f"Judges            : {USE_JUDGES}")
print(f"Mode              : {'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'}")

## Load the Phase 1 question set

In [ ]:
def list_phase1_runs():
    base = os.path.join(VOLUME_MOUNT, QUESTIONS_SUBDIR)
    return sorted(os.listdir(base)) if os.path.isdir(base) else []


runs = list_phase1_runs()
print(f"Phase 1 runs on volume: {runs}")

rid = QUESTION_RUN_ID or (runs[-1] if runs else None)
if not rid:
    raise RuntimeError("No Phase 1 runs found. Generate questions first.")


def norm_citation(q):
    """Phase 1 emitted 'citation' (str) on some batches, 'citations' (list) on others."""
    c = q.get("citation")
    if isinstance(c, str) and c.strip():
        return c
    cs = q.get("citations")
    if isinstance(cs, list) and cs:
        return "; ".join(str(x) for x in cs)
    return cs if isinstance(cs, str) else ""


qdir = os.path.join(VOLUME_MOUNT, QUESTIONS_SUBDIR, rid)
QUESTIONS = [json.loads(l) for l in open(os.path.join(qdir, "questions.jsonl"))]

rej_path = os.path.join(qdir, "rejected.jsonl")
if os.path.exists(rej_path):
    for q in (json.loads(l) for l in open(rej_path)):
        cite = norm_citation(q)
        if re.search(r"21\s*CFR\s*(1\.\d{3}|7\.\d{2}|810\.\d{2}|117\.\d{1,3})", cite) \
           and q["seed_section"] in cite:
            q.pop("_reject_reason", None)
            QUESTIONS.append(q)

seen = set()
QUESTIONS = [q for q in QUESTIONS if not (q["id"] in seen or seen.add(q["id"]))]

for q in QUESTIONS:
    q["citation"] = norm_citation(q)
    if q.get("difficulty") == "straightforward":
        q["difficulty"] = "recall"

print(f"\nUsing run: {rid}")
print(f"Questions: {len(QUESTIONS)}")
print(f"  by domain     : {dict(Counter(q['domain'] for q in QUESTIONS))}")
print(f"  by difficulty : {dict(Counter(q.get('difficulty') for q in QUESTIONS))}")
print(f"  missing cite  : {sum(1 for q in QUESTIONS if not q['citation'])}")

## Shared request helper

Returns the full envelope including `is_byok`, so billing questions get
answered by data instead of inference.

In [ ]:
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    in_bound = len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation."""
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def guarded_post(body, headers, timeout=REQUEST_TIMEOUT, purpose="call"):
    """One POST to OpenRouter. The attempt is reserved at its worst case before it is sent and settled after, so
    retries, exceptions and rejected calls are all counted. BudgetExceeded is raised before anything is sent."""
    body = openrouter_body(body)
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = requests.post(OPENROUTER_URL, headers=headers, json=body, timeout=timeout)
        if r.status_code == 200:
            try:
                usage = (r.json() or {}).get("usage") or None
            except Exception:
                usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


def call_openrouter(body, timeout=REQUEST_TIMEOUT, retries=MAX_RETRIES, label=""):
    """POST and normalize. Never raises."""
    last = None
    for attempt in range(retries):
        try:
            r = guarded_post(body, HEADERS, timeout=timeout, purpose=label or "call")

            if r.status_code in (429, 503):
                wait = r.headers.get("Retry-After")
                delay = float(wait) if wait and wait.isdigit() else 2 ** (attempt + 2)
                time.sleep(delay)
                last = f"HTTP {r.status_code}"
                continue

            if r.status_code >= 400:
                last = f"HTTP {r.status_code}: {r.text[:300]}"
                if 400 <= r.status_code < 500 and r.status_code != 429:
                    return {"ok": False, "error": last}   # client error won't fix itself
                time.sleep(2 ** attempt)
                continue

            d = r.json()
            if "error" in d:
                return {"ok": False, "error": str(d["error"])[:300]}

            msg = d["choices"][0]["message"]
            u = d.get("usage", {}) or {}
            return {
                "ok": True,
                "text": msg.get("content") or "",
                "model": d.get("model"),
                "provider": d.get("provider"),
                "is_byok": u.get("is_byok"),
                "cost": u.get("cost", 0) or 0,
                "prompt_tokens": u.get("prompt_tokens", 0),
                "completion_tokens": u.get("completion_tokens", 0),
                "reasoning_tokens": (u.get("completion_tokens_details") or {}).get("reasoning_tokens", 0),
                "error": None,
            }
        except BudgetExceeded:
            raise
        except Exception as e:
            last = f"{type(e).__name__}: {str(e)[:300]}"
            time.sleep(2 ** attempt)
    return {"ok": False, "error": last}


print("helper loaded")

## Capability discovery

Probes every model × effort pair with a throwaway call. Records what is
accepted, which provider served it, and whether your own key was used.
Costs a few cents and removes all guessing about effort support.

In [ ]:
def probe_config(cfg):
    model, effort, mode = cfg
    body = {
        "model": model,
        "provider": EVAL_PROVIDER,
        "messages": [{"role": "user", "content": "Reply with the single word OK."}],
        "max_tokens": 2000,
        "reasoning": {"effort": effort} if effort else {},
    }
    if mode:
        body["reasoning"]["mode"] = mode
    if not body["reasoning"]:
        body.pop("reasoning")

    res = call_openrouter(body, timeout=180, retries=2, label=f"{model}|{effort}")
    return {
        "model": model, "effort": effort, "mode": mode,
        "config_id": f"{model.split('/')[-1]}|{effort or 'default'}{'|pro' if mode else ''}",
        "supported": res["ok"],
        "provider": res.get("provider"),
        "is_byok": res.get("is_byok"),
        "error": res.get("error"),
    }


probe_list = [(m, e, None) for m in CANDIDATE_MODELS for e in CANDIDATE_EFFORTS]
if INCLUDE_PRO:
    probe_list += [(m, "xhigh", "pro") for m in PRO_VARIANTS]

print(f"Probing {len(probe_list)} model x effort combinations...\n", flush=True)

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    DISCOVERY = list(pool.map(probe_config, probe_list))

supported = [d for d in DISCOVERY if d["supported"]]
rejected = [d for d in DISCOVERY if not d["supported"]]

print(f"Supported: {len(supported)} / {len(DISCOVERY)}\n")
by_model = defaultdict(list)
for d in supported:
    by_model[d["model"]].append(d["effort"] + ("+pro" if d["mode"] else ""))
for m in CANDIDATE_MODELS:
    print(f"  {m:<26} {by_model.get(m, ['NONE'])}")

if rejected:
    print(f"\nRejected ({len(rejected)}):")
    for d in rejected[:12]:
        print(f"  {d['config_id']:<34} {str(d['error'])[:90]}")

byok_flags = Counter(d["is_byok"] for d in supported)
prov_flags = Counter(d["provider"] for d in supported)
print(f"\nis_byok across supported configs: {dict(byok_flags)}")
print(f"providers: {dict(prov_flags)}")
if False in byok_flags or None in byok_flags:
    print("\n*** Some configs did NOT use your key. Those bill OpenRouter credits, not OpenAI. ***")

## Judge probe

Same treatment for the two graders before they touch 2,000 answers.

In [ ]:
for name in USE_JUDGES:
    j = JUDGES[name]
    res = call_openrouter({
        "model": j["slug"],
        "provider": j["provider"],
        "messages": [{"role": "user", "content": "Reply with the single word OK."}],
        "max_tokens": 2000,
    }, timeout=180, retries=2)
    status = "OK  " if res["ok"] else "FAIL"
    print(f"[{status}] {name:<8} {j['slug']:<34} "
          f"provider={res.get('provider')} byok={res.get('is_byok')} "
          f"{'' if res['ok'] else str(res.get('error'))[:120]}")

## Build the run matrix

In [ ]:
CONFIGS = [
    {"config_id": d["config_id"], "model": d["model"],
     "effort": d["effort"], "mode": d["mode"]}
    for d in supported
]

EVAL_QUESTIONS = QUESTIONS
if SMOKE_TEST:
    CONFIGS = CONFIGS[:3]
    EVAL_QUESTIONS = random.sample(QUESTIONS, min(10, len(QUESTIONS)))

TASKS = [(c, q) for c in CONFIGS for q in EVAL_QUESTIONS]

print(f"Configs   : {len(CONFIGS)}")
print(f"Questions : {len(EVAL_QUESTIONS)}")
print(f"Eval calls: {len(TASKS)}")
print(f"Judge calls: {len(TASKS) * len(USE_JUDGES)}")
print(f"\nConfigs in this run:")
for c in CONFIGS:
    print(f"  {c['config_id']}")

## Answer generation

No section text and no web search — this measures what each model knows and
how it reasons, which is the thing being selected for.

In [ ]:
ANSWER_SYSTEM = """You are a food safety regulatory analyst. Answer the question directly and precisely.

Cite the specific CFR subsection your answer rests on. State any condition or exception that changes the answer. If the regulation does not address the question, say so rather than inferring a requirement.

Keep the answer under 200 words. No preamble."""


def run_eval_task(task):
    cfg, q = task
    body = {
        "model": cfg["model"],
        "provider": EVAL_PROVIDER,
        "messages": [
            {"role": "system", "content": ANSWER_SYSTEM},
            {"role": "user", "content": q["question"]},
        ],
        "max_tokens": 16000,
    }
    if cfg["effort"]:
        body["reasoning"] = {"effort": cfg["effort"]}
        if cfg["mode"]:
            body["reasoning"]["mode"] = cfg["mode"]

    res = call_openrouter(body, label=cfg["config_id"])
    return {
        "config_id": cfg["config_id"],
        "model": cfg["model"],
        "effort": cfg["effort"],
        "mode": cfg["mode"],
        "question_id": q["id"],
        "question": q["question"],
        "reference_answer": q["reference_answer"],
        "reference_citation": q["citation"],
        "domain": q["domain"],
        "difficulty": q.get("difficulty"),
        "answer": res.get("text", ""),
        "ok": res["ok"],
        "error": res.get("error"),
        "is_byok": res.get("is_byok"),
        "provider": res.get("provider"),
        "cost": res.get("cost", 0),
        "prompt_tokens": res.get("prompt_tokens", 0),
        "completion_tokens": res.get("completion_tokens", 0),
        "reasoning_tokens": res.get("reasoning_tokens", 0),
    }


run_id = RUN_ID
print(f"Run {run_id} — {len(TASKS)} eval calls, {MAX_WORKERS} workers", flush=True)

_t0 = time.time()
_done = [0]


def _tracked(task):
    r = run_eval_task(task)
    _done[0] += 1
    if _done[0] % 25 == 0:
        print(f"  {_done[0]}/{len(TASKS)}  ({time.time()-_t0:.0f}s)", flush=True)
    return r


with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    ANSWERS = list(pool.map(_tracked, TASKS))

eval_elapsed = time.time() - _t0
ok_answers = [a for a in ANSWERS if a["ok"] and a["answer"].strip()]
print(f"\nDone in {eval_elapsed:.0f}s")
print(f"  succeeded : {len(ok_answers)}/{len(ANSWERS)}")
print(f"  eval cost : ${sum(a['cost'] for a in ANSWERS):.3f}")
print(f"  is_byok   : {dict(Counter(a['is_byok'] for a in ANSWERS))}")

fails = [a for a in ANSWERS if not a["ok"]]
if fails:
    print(f"\n  failures by config: {dict(Counter(f['config_id'] for f in fails))}")
    print(f"  sample error: {fails[0]['error'][:200]}")

## Judging

Rubric, not a vague 1-10. Fabrication is an auto-zero — inventing a
requirement is categorically worse than missing one, and that is the failure
mode that matters for a model generating regulatory training data.

In [ ]:
JUDGE_SYSTEM = """You grade answers to US food safety regulation questions against a reference answer drawn from the actual CFR text.

Score these dimensions:

accuracy (0-4): Does the answer match the reference on substance?
  4 = fully correct
  3 = correct with a minor imprecision
  2 = partially correct, one material gap
  1 = mostly wrong
  0 = wrong

citation (0-3): Does it cite the correct subsection?
  3 = correct subsection
  2 = correct section, wrong or missing subsection
  1 = wrong section, right part
  0 = no citation, or a citation to something unrelated

completeness (0-3): Does it capture the conditions and exceptions in the reference?
  3 = all of them
  2 = the main one
  1 = mentions that conditions exist
  0 = none

fabrication (boolean): Does the answer state a requirement, deadline, threshold, or citation that is NOT supported by the reference? Judge this strictly. If true, the total is zero regardless of other scores.

Return JSON only:
{"accuracy": N, "citation": N, "completeness": N, "fabrication": true|false, "note": "one short sentence"}"""

JUDGE_TEMPLATE = """QUESTION:
{question}

REFERENCE ANSWER (ground truth, from CFR text):
{reference}
Reference citation: {ref_citation}

ANSWER TO GRADE:
{answer}

Grade it. JSON only."""


def judge_one(args):
    judge_name, ans = args
    j = JUDGES[judge_name]
    body = {
        "model": j["slug"],
        "provider": j["provider"],
        "messages": [
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": JUDGE_TEMPLATE.format(
                question=ans["question"],
                reference=ans["reference_answer"],
                ref_citation=ans["reference_citation"],
                answer=ans["answer"][:4000],
            )},
        ],
        "max_tokens": 4000,
    }
    res = call_openrouter(body, timeout=180)
    out = {"judge": judge_name, "config_id": ans["config_id"],
           "question_id": ans["question_id"], "cost": res.get("cost", 0)}
    if not res["ok"]:
        out.update({"score": None, "error": res.get("error")})
        return out
    try:
        txt = res["text"].replace("```json", "").replace("```", "").strip()
        b = txt.find("{")
        if b > 0:
            txt = txt[b:]
        g = json.loads(txt)
        total = 0 if g.get("fabrication") else (
            int(g.get("accuracy", 0)) + int(g.get("citation", 0)) + int(g.get("completeness", 0))
        )
        out.update({
            "accuracy": g.get("accuracy"), "citation": g.get("citation"),
            "completeness": g.get("completeness"), "fabrication": bool(g.get("fabrication")),
            "note": g.get("note", ""), "score": total, "error": None,
        })
    except Exception as e:
        out.update({"score": None, "error": f"parse: {type(e).__name__} :: {res['text'][:150]}"})
    return out


judge_tasks = [(jn, a) for jn in USE_JUDGES for a in ok_answers]
print(f"Judging {len(judge_tasks)} (answer x judge) pairs...", flush=True)

_j0 = time.time()
_jd = [0]


def _tracked_judge(t):
    r = judge_one(t)
    _jd[0] += 1
    if _jd[0] % 50 == 0:
        print(f"  {_jd[0]}/{len(judge_tasks)}  ({time.time()-_j0:.0f}s)", flush=True)
    return r


with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    GRADES = list(pool.map(_tracked_judge, judge_tasks))

judge_elapsed = time.time() - _j0
graded = [g for g in GRADES if g.get("score") is not None]
print(f"\nDone in {judge_elapsed:.0f}s")
print(f"  graded     : {len(graded)}/{len(GRADES)}")
print(f"  judge cost : ${sum(g.get('cost', 0) for g in GRADES):.3f}")
bad = [g for g in GRADES if g.get("score") is None]
if bad:
    print(f"  ungraded   : {len(bad)}  sample: {str(bad[0].get('error'))[:150]}")

## Scoring

Bootstrapped 95% CIs. Judge agreement is reported first — if the two judges
disagree badly, the ranking below is a judge artifact, not a finding.

In [ ]:
def bootstrap_ci(vals, n=2000, alpha=0.05):
    if len(vals) < 2:
        return (None, None)
    rng = random.Random(SEED)
    means = []
    for _ in range(n):
        s = [vals[rng.randrange(len(vals))] for _ in range(len(vals))]
        means.append(sum(s) / len(s))
    means.sort()
    return (round(means[int(n * alpha / 2)], 3), round(means[int(n * (1 - alpha / 2))], 3))


# --- judge agreement ---
paired = defaultdict(dict)
for g in graded:
    paired[(g["config_id"], g["question_id"])][g["judge"]] = g["score"]

both = [(v[USE_JUDGES[0]], v[USE_JUDGES[1]])
        for v in paired.values() if len(USE_JUDGES) > 1 and len(v) == 2]

print("=" * 74)
if both:
    diffs = [abs(a - b) for a, b in both]
    exact = sum(1 for d in diffs if d == 0) / len(diffs)
    within1 = sum(1 for d in diffs if d <= 1) / len(diffs)
    ma, mb = statistics.mean([x[0] for x in both]), statistics.mean([x[1] for x in both])
    try:
        corr = statistics.correlation([x[0] for x in both], [x[1] for x in both])
    except Exception:
        corr = None
    print("JUDGE AGREEMENT")
    print(f"  paired gradings   : {len(both)}")
    print(f"  exact match       : {exact:.1%}")
    print(f"  within 1 point    : {within1:.1%}")
    print(f"  mean |difference| : {statistics.mean(diffs):.2f} of 10")
    print(f"  correlation       : {corr if corr is None else round(corr,3)}")
    print(f"  {USE_JUDGES[0]} mean {ma:.2f}  |  {USE_JUDGES[1]} mean {mb:.2f}")
    if corr is not None and corr < 0.5:
        print("  *** Low agreement. Treat the ranking below as unreliable. ***")
else:
    print("JUDGE AGREEMENT: single judge — no cross-check available.")

# --- per-config scores ---
by_cfg = defaultdict(list)
fab_by_cfg = defaultdict(int)
for g in graded:
    by_cfg[g["config_id"]].append(g["score"])
    if g.get("fabrication"):
        fab_by_cfg[g["config_id"]] += 1

cost_by_cfg = defaultdict(float)
rtok_by_cfg = defaultdict(int)
for a in ANSWERS:
    cost_by_cfg[a["config_id"]] += a["cost"]
    rtok_by_cfg[a["config_id"]] += a["reasoning_tokens"]

rows = []
for cid, scores in by_cfg.items():
    lo, hi = bootstrap_ci(scores)
    rows.append({
        "config_id": cid,
        "n": len(scores),
        "mean": round(sum(scores) / len(scores), 3),
        "ci_low": lo, "ci_high": hi,
        "fabrications": fab_by_cfg[cid],
        "cost": round(cost_by_cfg[cid], 4),
        "reasoning_tokens": rtok_by_cfg[cid],
    })
rows.sort(key=lambda r: r["mean"], reverse=True)

print("\n" + "=" * 74)
print("RANKING (max score 10; fabrication forces 0)")
print("=" * 74)
print(f"{'config':<32}{'mean':>7}{'95% CI':>16}{'fab':>5}{'$':>9}")
for r in rows:
    ci = f"[{r['ci_low']}, {r['ci_high']}]" if r["ci_low"] is not None else "—"
    print(f"{r['config_id']:<32}{r['mean']:>7}{ci:>16}{r['fabrications']:>5}{r['cost']:>9.4f}")

if rows:
    top = rows[0]
    overlapping = [r for r in rows[1:]
                   if r["ci_high"] is not None and top["ci_low"] is not None
                   and r["ci_high"] >= top["ci_low"]]
    print(f"\nLeader: {top['config_id']} ({top['mean']})")
    print(f"Configs whose CI overlaps the leader: {len(overlapping)} of {len(rows)-1}")
    if len(overlapping) > len(rows) * 0.6:
        print("Most of the field is statistically tied. More questions would be")
        print("needed to separate them.")

    print("\nScore per dollar (cheapest config within the leader's CI):")
    contenders = [top] + overlapping
    contenders.sort(key=lambda r: r["cost"])
    for r in contenders[:5]:
        print(f"  {r['config_id']:<32} mean {r['mean']:<7} ${r['cost']:.4f}")

## Write to volume

In [ ]:
outdir = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, run_id)
os.makedirs(outdir, exist_ok=True)


def dump(name, rows_):
    p = os.path.join(outdir, name)
    with open(p, "w") as f:
        for r in rows_:
            f.write(json.dumps(r) + "\n")
    return p


p_ans = dump("answers.jsonl", ANSWERS)
p_grd = dump("grades.jsonl", GRADES)
p_dis = dump("discovery.jsonl", DISCOVERY)
p_res = dump("results.jsonl", rows)

manifest = {
    "run_id": run_id,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "smoke_test": SMOKE_TEST,
    "question_run_id": rid,
    "questions_used": len(EVAL_QUESTIONS),
    "configs_discovered": len(supported),
    "configs_run": len(CONFIGS),
    "eval_calls": len(ANSWERS),
    "eval_ok": len(ok_answers),
    "judges": USE_JUDGES,
    "judge_calls": len(GRADES),
    "judge_ok": len(graded),
    "judge_agreement": {
        "paired": len(both),
        "exact_pct": round(exact, 4) if both else None,
        "within1_pct": round(within1, 4) if both else None,
        "correlation": round(corr, 4) if both and corr is not None else None,
    },
    "eval_cost_usd": round(sum(a["cost"] for a in ANSWERS), 4),
    "judge_cost_usd": round(sum(g.get("cost", 0) for g in GRADES), 4),
    "eval_is_byok": {str(k): v for k, v in Counter(a["is_byok"] for a in ANSWERS).items()},
    "elapsed_seconds": {"eval": round(eval_elapsed, 1), "judge": round(judge_elapsed, 1)},
    "ranking": rows,
    "files": {"answers": p_ans, "grades": p_grd,
              "discovery": p_dis, "results": p_res},
}
with open(os.path.join(outdir, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Total cost: ${manifest['eval_cost_usd'] + manifest['judge_cost_usd']:.3f}")
print(f"Output:     {outdir}")

## Inspect

Read a few graded answers before trusting the ranking. A judge that likes
verbosity or penalizes a correct-but-differently-worded answer will show up
here and nowhere else.

In [ ]:
def show(config_id=None, n=3, only_low=False):
    pool_ = [g for g in graded if config_id is None or g["config_id"] == config_id]
    if only_low:
        pool_ = sorted(pool_, key=lambda g: g["score"])
    for g in pool_[:n]:
        a = next((x for x in ok_answers
                  if x["config_id"] == g["config_id"] and x["question_id"] == g["question_id"]), None)
        if not a:
            continue
        print("=" * 74)
        print(f"{g['config_id']} | {g['question_id']} | judge={g['judge']} | score={g['score']}/10")
        print(f"  acc={g.get('accuracy')} cite={g.get('citation')} "
              f"comp={g.get('completeness')} fab={g.get('fabrication')}")
        print(f"  note: {g.get('note')}")
        print(f"\nQ: {a['question'][:300]}")
        print(f"\nREF: {a['reference_answer'][:400]}")
        print(f"\nGOT: {a['answer'][:400]}")


if SMOKE_TEST and graded:
    print("Lowest-scoring answers — check these for judge error:\n")
    show(n=3, only_low=True)